# XGBoost

### 서울

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[19:10:26] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9820 | MAE: 4,400 | RMSE: 7,638 | MAPE: 4.23% | MdAPE: 3.22% | RMSLE: 0.0591
------------------------------
FINAL TEST RESULT: R2: 0.9225 | MAE: 10,819 | RMSE: 17,930 | MAPE: 9.28% | MdAPE: 6.46% | RMSLE: 0.1235


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 기존 GRU 코드와 완전 동일한 출력 형식
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:40:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9924 | MAE: 3,032 | RMSE: 5,646 | MAPE: 2.51% | MdAPE: 1.67% | RMSLE: 0.0400
------------------------------
FINAL TEST RESULT: R2: 0.9723 | MAE: 7,814 | RMSE: 11,094 | MAPE: 7.07% | MdAPE: 5.62% | RMSLE: 0.0842


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:42:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9904 | MAE: 3,223 | RMSE: 6,324 | MAPE: 2.55% | MdAPE: 1.71% | RMSLE: 0.0406
------------------------------
FINAL TEST RESULT: R2: 0.9709 | MAE: 7,623 | RMSE: 11,358 | MAPE: 6.94% | MdAPE: 5.40% | RMSLE: 0.0840


### 부산

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:43:31] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9736 | MAE: 1,571 | RMSE: 3,281 | MAPE: 5.04% | MdAPE: 3.71% | RMSLE: 0.0706
------------------------------
FINAL TEST RESULT: R2: 0.8938 | MAE: 3,997 | RMSE: 8,716 | MAPE: 9.33% | MdAPE: 6.44% | RMSLE: 0.1388


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:43:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9920 | MAE: 1,174 | RMSE: 2,704 | MAPE: 2.70% | MdAPE: 2.12% | RMSLE: 0.0360
------------------------------
FINAL TEST RESULT: R2: 0.9708 | MAE: 2,574 | RMSE: 5,236 | MAPE: 6.14% | MdAPE: 4.53% | RMSLE: 0.0784


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:44:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9939 | MAE: 1,181 | RMSE: 2,302 | MAPE: 3.05% | MdAPE: 2.38% | RMSLE: 0.0397
------------------------------
FINAL TEST RESULT: R2: 0.9735 | MAE: 2,391 | RMSE: 4,945 | MAPE: 5.86% | MdAPE: 4.26% | RMSLE: 0.0766


### 대구

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:44:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9855 | MAE: 940 | RMSE: 1,625 | MAPE: 3.83% | MdAPE: 2.81% | RMSLE: 0.0555
------------------------------
FINAL TEST RESULT: R2: 0.9443 | MAE: 2,055 | RMSE: 3,280 | MAPE: 7.95% | MdAPE: 5.68% | RMSLE: 0.1054


In [ ]:
import os

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:44:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9939 | MAE: 623 | RMSE: 1,264 | MAPE: 2.08% | MdAPE: 1.71% | RMSLE: 0.0269
------------------------------
FINAL TEST RESULT: R2: 0.9773 | MAE: 1,467 | RMSE: 2,313 | MAPE: 5.75% | MdAPE: 4.29% | RMSLE: 0.0729


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:44:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9950 | MAE: 663 | RMSE: 1,123 | MAPE: 2.29% | MdAPE: 1.90% | RMSLE: 0.0291
------------------------------
FINAL TEST RESULT: R2: 0.9797 | MAE: 1,369 | RMSE: 2,170 | MAPE: 5.35% | MdAPE: 3.85% | RMSLE: 0.0696


### 대전

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:45:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9726 | MAE: 1,569 | RMSE: 2,893 | MAPE: 5.19% | MdAPE: 4.11% | RMSLE: 0.0710
------------------------------
FINAL TEST RESULT: R2: 0.9158 | MAE: 3,261 | RMSE: 5,803 | MAPE: 9.52% | MdAPE: 7.38% | RMSLE: 0.1253


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:45:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9951 | MAE: 844 | RMSE: 1,600 | MAPE: 2.11% | MdAPE: 1.66% | RMSLE: 0.0283
------------------------------
FINAL TEST RESULT: R2: 0.9813 | MAE: 1,906 | RMSE: 2,988 | MAPE: 5.60% | MdAPE: 3.62% | RMSLE: 0.0762


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:45:51] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9937 | MAE: 988 | RMSE: 1,767 | MAPE: 2.85% | MdAPE: 2.33% | RMSLE: 0.0362
------------------------------
FINAL TEST RESULT: R2: 0.9826 | MAE: 1,763 | RMSE: 2,849 | MAPE: 5.53% | MdAPE: 3.60% | RMSLE: 0.0755


### 광주

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략 (기존 코드 형식 모사)
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:46:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9865 | MAE: 891 | RMSE: 1,359 | MAPE: 4.71% | MdAPE: 3.53% | RMSLE: 0.0636
------------------------------
FINAL TEST RESULT: R2: 0.9290 | MAE: 2,129 | RMSE: 3,736 | MAPE: 8.34% | MdAPE: 5.93% | RMSLE: 0.1202


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:46:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9935 | MAE: 558 | RMSE: 1,241 | MAPE: 2.30% | MdAPE: 1.85% | RMSLE: 0.0312
------------------------------
FINAL TEST RESULT: R2: 0.9681 | MAE: 1,435 | RMSE: 2,852 | MAPE: 5.34% | MdAPE: 3.50% | RMSLE: 0.0753


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링 (XGBoost용 2D 변환 포함)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (XGBoost) - 파라미터 및 출력 형식 통일
print(f"{'='*30}\nSTART: XGBOOST (SAMPLE-WISE SPLIT)\n{'='*30}")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 6. 학습
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    early_stopping_rounds=20,
    verbose=False # Epoch마다 찍히는 로그 생략
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: XGBOOST (SAMPLE-WISE SPLIT)
[18:46:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "verbose" } are not used.

Validation 결과 | R2: 0.9948 | MAE: 529 | RMSE: 1,078 | MAPE: 2.22% | MdAPE: 1.75% | RMSLE: 0.0303
------------------------------
FINAL TEST RESULT: R2: 0.9714 | MAE: 1,392 | RMSE: 2,680 | MAPE: 5.26% | MdAPE: 3.56% | RMSLE: 0.0737
